In [ ]:
cnivel  = 'X:atrisk_fire1/topografia/cnivel/cnvl_centro.shp'
col     = 'ELEVATION'
bb_dem  = 'X:atrisk_fire1/reflmt/roi_centro_dem.shp'
bb_tin  = None
out_dem = 'X:atrisk_fire1/topografia/centro/centro_dem50x50.tif'
epsg    = 3763
csize   = 50

from glass.esri.rst.surf import dem_from_tin

dem_from_tin(
    cnivel, col, bb_dem, csize,
    out_dem, bbox_tin=bb_tin, prj=epsg
)

In [ ]:
from glass.esri.rst.surf import slope, aspect

dem    = 'X:atrisk_fire1/topografia/centro/centro_filldem.tif'
slpr = 'X:atrisk_fire1/topografia/coimbra/cmb_slope.tif'
aspec = 'X:atrisk_fire1/topografia/centro/centro_aspect.tif'

#slope(dem, slpr, "DEGREE")
aspect(dem, aspec, reclass=True)

In [ ]:
"""
TPI
"""

dem    = 'X:atrisk_fire1/topografia/concse/se_elev.tif'
window = (5, 5)
tpi    = 'X:atrisk_fire1/topografia/concse/se_tpi.tif'

import os
from glass.esri.rst.neigh import focal_statistics
from glass.esri.rst.alg import rstcalc
from glass.pys.oss import mkdir

ws = mkdir(os.path.dirname(tpi), timerand=True, overwrite=True)

orst, ofocal = focal_statistics(
    dem, os.path.join(ws, 'focal_dem.tif'),
    "Rectangle", window, "MEAN"
)

_, reslyr = rstcalc(
    [dem, ofocal], ['d', 'f'],
    'd - f', tpi,
    template=dem
)

In [ ]:
"""
Produce a MDT for each cell in vec frid file
"""

alti = r'C:\gwork\fireloc\datasets\mdt_m888\alti_pt25k_3763.shp'
ref  = r'C:\gwork\fireloc\datasets\ref\pt_ref_grid.shp'
outFld = r'C:\gwork\fireloc\datasets\mdt_m888'
cell_id = 'cellid'

cell_width = 3000
cell_height= 3000

import os; import datetime as dt
from glass.rd          import tbl_to_obj
from glass.gp.prox.bfing  import df_buffer_extent
from glass.gp.cnv import coords_to_boundary
from glass.pys.oss import create_folder
from glass.pys.oss import fld_exists

from gesri.df.gop.ovlay import clip

epsg = 3763

grid_df = tbl_to_obj(ref)

grid_df = df_buffer_extent(grid_df, epsg, cell_width, mantainOriginalGeom=True)

In [ ]:
def clip_r(row):
    fld_path = os.path.join(outFld, 'data_' + str(row[cell_id]))
    
    isFld = fld_exists(fld_path)
    if not isFld:
        fld = create_folder(fld_path)
    else:
        continue
    
    # Get Lmt Shape
    left, bottom, right, top = row.old_geom.bounds
    lmt = coords_to_boundary((left, top), (right, bottom), epsg, os.path.join(
        fld, f'lmt_{str(row[cell_id])}.shp'
    ))
    
    # Get Extra Lmt
    left, bottom, right, top = row.geometry.bounds
    extra_lmt = coords_to_boundary((left, top), (right, bottom), epsg, os.path.join(
        fld, f'extra_lmt_{str(row[cell_id])}.shp'
    ))
    
    # Clip Altimetry
    alti_clp = clip(alti, extra_lmt, os.path.join(
        fld, f'alti_{str(row[cell_id])}.shp'
    ))
    
    return row

time_a = dt.datetime.now().replace(microsecond=0)
grid_df.apply(lambda x: clip_r(x), axis=1)
time_b = dt.datetime.now().replace(microsecond=0)
print(time_b - time_a)